# Plotly Cached Potential Visualization

This notebook is built around the existing `calculator` and `run_evaluation` code.

It uses two cache layers:

- The existing `run_evaluation` JSON cache for the combined-run summary and the default `W_R_T` values.
- A dedicated Plotly scan cache for the extra parameter sweeps needed for interactive `V(R)` and `r_0` stability plots.

The scan cache is written once and reused on later notebook runs, so the notebook does not recompute the same parameter scans every time.


In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import run_evaluation
from calculator import Calculator, cornell_potential_ansatz

pio.templates.default = "plotly_white"


In [ ]:
RUN_PATH = Path("../data/20260316/42")

# Choose exactly one data source mode: "combined" or "single".
DATA_MODE = "combined"

FORCE_SCAN_CACHE_REBUILD = False
BUILD_SCAN_CACHE_IF_MISSING = False
LOAD_WORKERS = 1

T_MIN_VALUES = list(range(1, 10))
T_MAX_VALUES = [None, 20]

R0_T_MIN = 5
R0_T_MAX = None
R_MIN_VALUES = list(range(1, 6))
TARGET_FORCE = run_evaluation.DEFAULT_SOMMER_TARGET

OUTPUT_DIR = Path.cwd() / "interactive_html"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_PATH = {RUN_PATH.resolve()}")
print(f"DATA_MODE = {DATA_MODE}")
print(f"HTML output = {OUTPUT_DIR}")


In [ ]:
RUN_PATH = RUN_PATH.resolve()

if DATA_MODE not in {"combined", "single"}:
    raise ValueError("DATA_MODE must be either 'combined' or 'single'")

combine_equivalent_runs = DATA_MODE == "combined"

if combine_equivalent_runs:
    analysis_id, grouped_paths = run_evaluation._discover_equivalent_runs(str(RUN_PATH))
else:
    analysis_id = f"run_{run_evaluation.get_run_id(str(RUN_PATH))}"
    grouped_paths = [str(RUN_PATH)]

summary_cache = run_evaluation.load_cached_result(analysis_id)
if summary_cache is None:
    raise FileNotFoundError(
        f"No run_evaluation cache found for {analysis_id}. "
        "Run run_evaluation.get_or_calculate(...) once before using this notebook."
    )

plot_cache_path = run_evaluation.CALC_RESULT_BASE / f"{analysis_id}__plotly_potential_v1.json"
plot_output_dir = OUTPUT_DIR / analysis_id
plot_output_dir.mkdir(parents=True, exist_ok=True)

print(f"analysis_id = {analysis_id}")
print(f"mode run count = {len(grouped_paths)}")
print(f"summary cache = {run_evaluation.get_result_path(analysis_id)}")
print(f"scan cache = {plot_cache_path}")


In [ ]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def _window_sort_key(window):
    t_min, t_max = window
    if t_max is None:
        return (int(t_min), math.inf)
    return (int(t_min), int(t_max))


def _window_label(t_min, t_max):
    return f"t_min={t_min}, t_max={'None' if t_max is None else t_max}"


def _load_scan_cache(path: Path):
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def _save_scan_cache(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=_json_default)
    print(f"saved scan cache: {path}")


def _build_scan_cache():
    combined_w_temp, aggregation = run_evaluation._load_combined_w_temp(
        grouped_paths,
        verbose=True,
        prefix=f"[{RUN_PATH.name}]",
        load_workers=LOAD_WORKERS,
    )
    if combined_w_temp is None:
        raise RuntimeError("No combined W_temp data could be loaded for this run group")

    block_size = int(summary_cache.get("block_size", 1) or 1)
    calc = Calculator(
        combined_w_temp,
        n_bootstrap=run_evaluation.DEFAULT_N_BOOTSTRAP,
        step_size=block_size,
    )

    unique_rs = [int(r) for r in calc.get_unique_Rs()]
    unique_ts = [int(t) for t in calc.get_unique_Ts()]

    windows = []
    for t_min in T_MIN_VALUES:
        for t_max in T_MAX_VALUES:
            if t_max is not None and int(t_max) < int(t_min):
                continue
            windows.append((int(t_min), None if t_max is None else int(t_max)))
    windows = sorted(set(windows), key=_window_sort_key)

    v_records = []
    for t_min, t_max in windows:
        for r_val in unique_rs:
            try:
                v_var = calc.get_variable("V_R", R=int(r_val), t_min=int(t_min), t_max=t_max)
            except Exception:
                continue

            value = v_var.get()
            if value is None or not np.isfinite(value):
                continue

            err = v_var.err()
            fit_c = v_var.parameters.get("fit_C")
            v_records.append({
                "R": int(r_val),
                "t_min": int(t_min),
                "t_max": t_max,
                "value": float(value),
                "err": float(err) if err is not None and np.isfinite(err) else None,
                "fit_C": float(fit_c) if fit_c is not None and np.isfinite(fit_c) else None,
            })

    r0_records = []
    curve_records = []
    for r_min in [int(v) for v in R_MIN_VALUES]:
        try:
            r0_var = calc.get_variable(
                "r0",
                t_min=int(R0_T_MIN),
                t_max=R0_T_MAX,
                target_force=float(TARGET_FORCE),
                r_min=int(r_min),
            )
        except Exception:
            continue

        r0_value = r0_var.get()
        if r0_value is None or not np.isfinite(r0_value):
            continue

        r0_err = r0_var.err()
        params = r0_var.parameters.get("cornell_params", {}) or {}
        A = params.get("A")
        sigma = params.get("sigma")
        B = params.get("B")

        r0_records.append({
            "r_min": int(r_min),
            "r0": float(r0_value),
            "err": float(r0_err) if r0_err is not None and np.isfinite(r0_err) else None,
            "A": float(A) if A is not None and np.isfinite(A) else None,
            "sigma": float(sigma) if sigma is not None and np.isfinite(sigma) else None,
            "B": float(B) if B is not None and np.isfinite(B) else None,
            "t_min": int(R0_T_MIN),
            "t_max": R0_T_MAX,
            "target_force": float(TARGET_FORCE),
        })

        if A is None or sigma is None or B is None:
            continue

        fit_rs = [int(r) for r in unique_rs if int(r) >= int(r_min)]
        if not fit_rs:
            continue

        for r_val in fit_rs:
            try:
                v_var = calc.get_variable("V_R", R=int(r_val), t_min=int(R0_T_MIN), t_max=R0_T_MAX)
                v_value = v_var.get()
                v_err = v_var.err()
            except Exception:
                v_value = None
                v_err = None

            curve_records.append({
                "kind": "point",
                "r_min": int(r_min),
                "R": float(r_val),
                "V_fit": float(cornell_potential_ansatz(float(r_val), float(A), float(sigma), float(B))),
                "V": float(v_value) if v_value is not None and np.isfinite(v_value) else None,
                "err": float(v_err) if v_err is not None and np.isfinite(v_err) else None,
            })

        r_grid = np.linspace(min(fit_rs), max(fit_rs), 300)
        for r_value in r_grid:
            curve_records.append({
                "kind": "curve",
                "r_min": int(r_min),
                "R": float(r_value),
                "V_fit": float(cornell_potential_ansatz(float(r_value), float(A), float(sigma), float(B))),
                "V": None,
                "err": None,
            })

    return {
        "version": 1,
        "analysis_id": analysis_id,
        "path": str(RUN_PATH),
        "grouped_paths": grouped_paths,
        "aggregation": aggregation,
        "block_size": block_size,
        "t_windows": [
            {"t_min": int(t_min), "t_max": t_max}
            for t_min, t_max in windows
        ],
        "v_r_scan": v_records,
        "r0_scan": r0_records,
        "cornell_curves": curve_records,
    }


The next cell loads the dedicated scan cache if it already exists.

If the file is missing and `BUILD_SCAN_CACHE_IF_MISSING = True`, it builds the scan cache once from the same combined `W_temp` data path used by `run_evaluation`, then writes it back to `../data/calcResult/` for later runs.


In [ ]:
scan_cache = None if FORCE_SCAN_CACHE_REBUILD else _load_scan_cache(plot_cache_path)

if scan_cache is None:
    if not BUILD_SCAN_CACHE_IF_MISSING:
        raise FileNotFoundError(
            f"Scan cache missing: {plot_cache_path}. "
            "Set BUILD_SCAN_CACHE_IF_MISSING = True to create it once."
        )
    scan_cache = _build_scan_cache()
    _save_scan_cache(plot_cache_path, scan_cache)
else:
    print(f"loaded scan cache: {plot_cache_path}")


In [ ]:
def _wrt_dataframe(cache: dict) -> pd.DataFrame:
    rows = []
    for key, value in cache.get("W_R_T", {}).items():
        r_text, t_text = key.split(",")
        rows.append({
            "R": int(r_text),
            "T": int(t_text),
            "value": float(value),
        })
    return pd.DataFrame(rows).sort_values(["R", "T"]).reset_index(drop=True)


wrt_df = _wrt_dataframe(summary_cache)
v_scan_df = pd.DataFrame(scan_cache.get("v_r_scan", []))
r0_scan_df = pd.DataFrame(scan_cache.get("r0_scan", []))
cornell_df = pd.DataFrame(scan_cache.get("cornell_curves", []))

print(f"W_R_T rows: {len(wrt_df)}")
print(f"V(R) scan rows: {len(v_scan_df)}")
print(f"r0 scan rows: {len(r0_scan_df)}")
print(f"Cornell curve rows: {len(cornell_df)}")


In [ ]:
def save_and_show(fig: go.Figure, name: str):
    html_path = plot_output_dir / name
    fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
    fig.show()
    print(f"saved html: {html_path}")
    return html_path


def make_wrt_plot(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("No W_R_T data found in the summary cache")

    r_values = sorted(df["R"].unique())
    fig = go.Figure()
    for idx, r_value in enumerate(r_values):
        subset = df[df["R"] == r_value]
        fig.add_trace(
            go.Scatter(
                x=subset["T"],
                y=subset["value"],
                mode="lines+markers",
                name=f"R = {r_value}",
                visible=idx == 0,
            )
        )

    buttons = []
    for idx, r_value in enumerate(r_values):
        visible = [False] * len(r_values)
        visible[idx] = True
        buttons.append({
            "label": f"R = {r_value}",
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"W(R,T) for R = {r_value}"},
            ],
        })

    fig.update_layout(
        title=f"W(R,T) for R = {r_values[0]}",
        xaxis_title="T",
        yaxis_title="W(R,T)",
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.02,
            "xanchor": "left",
            "y": 1.0,
            "yanchor": "top",
        }],
        margin={"l": 60, "r": 220, "t": 70, "b": 60},
    )
    return fig


def make_vr_plot(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("No V(R) scan cache data available")

    window_table = (
        df[["t_min", "t_max"]]
        .drop_duplicates()
        .assign(_sort=lambda frame: frame.apply(lambda row: _window_sort_key((row["t_min"], row["t_max"])), axis=1))
        .sort_values("_sort")
        .drop(columns="_sort")
        .reset_index(drop=True)
    )
    windows = [tuple(row) for row in window_table.itertuples(index=False, name=None)]

    fig = go.Figure()
    for idx, (t_min, t_max) in enumerate(windows):
        subset = df[(df["t_min"] == t_min) & (df["t_max"].isna() if t_max is None else df["t_max"] == t_max)].sort_values("R")
        error_y = None
        if "err" in subset.columns and subset["err"].notna().any():
            error_y = {
                "type": "data",
                "array": subset["err"].fillna(0.0),
                "visible": True,
            }
        fig.add_trace(
            go.Scatter(
                x=subset["R"],
                y=subset["value"],
                mode="lines+markers",
                name=_window_label(t_min, t_max),
                visible=idx == 0,
                error_y=error_y,
            )
        )

    buttons = []
    for idx, (t_min, t_max) in enumerate(windows):
        visible = [False] * len(windows)
        visible[idx] = True
        buttons.append({
            "label": _window_label(t_min, t_max),
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"V(R) for {_window_label(t_min, t_max)}"},
            ],
        })

    first_t_min, first_t_max = windows[0]
    fig.update_layout(
        title=f"V(R) for {_window_label(first_t_min, first_t_max)}",
        xaxis_title="R",
        yaxis_title="V(R)",
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.02,
            "xanchor": "left",
            "y": 1.0,
            "yanchor": "top",
        }],
        margin={"l": 60, "r": 260, "t": 70, "b": 60},
    )
    return fig


def make_r0_stability_plot(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("No r0 scan cache data available")

    ordered = df.sort_values("r_min")
    error_y = None
    if "err" in ordered.columns and ordered["err"].notna().any():
        error_y = {
            "type": "data",
            "array": ordered["err"].fillna(0.0),
            "visible": True,
        }

    fig = go.Figure(
        data=[
            go.Scatter(
                x=ordered["r_min"],
                y=ordered["r0"],
                mode="lines+markers",
                name="r0",
                error_y=error_y,
            )
        ]
    )
    fig.update_layout(
        title=f"r0 stability for t_min = {ordered['t_min'].iloc[0]}, t_max = {ordered['t_max'].iloc[0]}",
        xaxis_title="r_min",
        yaxis_title="r0/a",
        margin={"l": 60, "r": 40, "t": 70, "b": 60},
    )
    return fig


def make_cornell_plot(curves: pd.DataFrame, r0_scan: pd.DataFrame) -> go.Figure:
    if curves.empty or r0_scan.empty:
        raise ValueError("Cornell curve cache data is missing")

    r_min_values = sorted(r0_scan["r_min"].unique())
    fig = go.Figure()

    point_subset = curves[(curves["kind"] == "point") & curves["V"].notna()].sort_values("R")
    if not point_subset.empty:
        base_points = point_subset.drop_duplicates(subset=["R", "V", "err"]).sort_values("R")
        error_y = None
        if base_points["err"].notna().any():
            error_y = {
                "type": "data",
                "array": base_points["err"].fillna(0.0),
                "visible": True,
            }
        fig.add_trace(
            go.Scatter(
                x=base_points["R"],
                y=base_points["V"],
                mode="markers",
                name="V(R) data",
                visible=True,
                error_y=error_y,
            )
        )

    fit_trace_indices = {}
    marker_trace_indices = {}
    for idx, r_min in enumerate(r_min_values):
        curve_subset = curves[(curves["kind"] == "curve") & (curves["r_min"] == r_min)].sort_values("R")
        fit_trace_indices[r_min] = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=curve_subset["R"],
                y=curve_subset["V_fit"],
                mode="lines",
                name=f"Cornell fit (r_min={r_min})",
                visible=idx == 0,
            )
        )

        row = r0_scan[r0_scan["r_min"] == r_min].iloc[0]
        marker_y = None
        if pd.notna(row.get("A")) and pd.notna(row.get("sigma")) and pd.notna(row.get("B")):
            marker_y = cornell_potential_ansatz(float(row["r0"]), float(row["A"]), float(row["sigma"]), float(row["B"]))
        marker_trace_indices[r_min] = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=[row["r0"]],
                y=[marker_y],
                mode="markers",
                marker={"size": 10, "symbol": "diamond"},
                name=f"r0 (r_min={r_min})",
                visible=idx == 0,
            )
        )

    total_traces = len(fig.data)
    has_base_points = total_traces > 0 and fig.data[0].name == "V(R) data"

    buttons = []
    for r_min in r_min_values:
        visible = [False] * total_traces
        if has_base_points:
            visible[0] = True
        visible[fit_trace_indices[r_min]] = True
        visible[marker_trace_indices[r_min]] = True
        r0_value = r0_scan.loc[r0_scan["r_min"] == r_min, "r0"].iloc[0]
        buttons.append({
            "label": f"r_min = {r_min}",
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"Cornell fit for r_min = {r_min} (r0/a = {r0_value:.4f})"},
            ],
        })

    first_r_min = r_min_values[0]
    first_r0 = r0_scan.loc[r0_scan["r_min"] == first_r_min, "r0"].iloc[0]
    fig.update_layout(
        title=f"Cornell fit for r_min = {first_r_min} (r0/a = {first_r0:.4f})",
        xaxis_title="R",
        yaxis_title="V(R)",
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.02,
            "xanchor": "left",
            "y": 1.0,
            "yanchor": "top",
        }],
        margin={"l": 60, "r": 260, "t": 70, "b": 60},
    )
    return fig


In [ ]:
wrt_fig = make_wrt_plot(wrt_df)
save_and_show(wrt_fig, "wrt_by_R.html")

vr_fig = make_vr_plot(v_scan_df)
save_and_show(vr_fig, "potential_by_t_window.html")

r0_fig = make_r0_stability_plot(r0_scan_df)
save_and_show(r0_fig, "r0_vs_r_min.html")

cornell_fig = make_cornell_plot(cornell_df, r0_scan_df)
save_and_show(cornell_fig, "cornell_fit_by_r_min.html")


If you want different scan ranges, update:

- `T_MIN_VALUES`
- `T_MAX_VALUES`
- `R0_T_MIN`
- `R0_T_MAX`
- `R_MIN_VALUES`

Then set `FORCE_SCAN_CACHE_REBUILD = True` for one run to refresh the scan cache.
